In [0]:
%sql
use catalog ext_cat;

In [0]:
landing_zone =  '/Volumes/ext_cat/default/raw'
orders_data = landing_zone + '/ordershistory'
checkpoint_path =  landing_zone + '/orders_checkpoint'

In [0]:
ordersdf= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", checkpoint_path) \
  .option("cloudFiles.inferSchema", "true") \
  .option("cloudFiles.inferColumnTypes", 'true') \
  .load(orders_data) \
  

In [0]:
%sql
drop table if exists ext_cat.default.orders;

streaming job runs continously until failure or manual stop and keep looking for file every 10 secs and loads them automatically.
Job failed when in encountered a new file with new column, 
then we need to redo read stream and rerun write steam

In [0]:
ordersdf.writeStream \
    .format('delta') \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .toTable('ext_cat.default.orders')

In [0]:
%sql
select * from orders

order_id,order_date,customer_id,order_status,_rescued_data,new_col
10101,null,null,Delivered,"{""order_date"":""25-08-2025"",""customer_id"":""ABC"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",1
10102,null,2105,Shipped,"{""order_date"":""26-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",2
10103,null,2102,Processing,"{""order_date"":""27-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",3
10104,null,2108,Cancelled,"{""order_date"":""28-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",4
10105,null,2101,Delivered,"{""order_date"":""30-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",5
10106,null,2110,Shipped,"{""order_date"":""01-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",6
10107,null,2104,Processing,"{""order_date"":""02-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",7
10108,null,2107,Pending,"{""order_date"":""03-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",8
10109,null,1203,Pending,"{""order_date"":""05-09-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders5.csv""}",1
10101,null,2101,Delivered,"{""order_date"":""25-08-2025"",""_file_path"":""/Volumes/ext_cat/default/raw/ordershistory/orders3.csv""}",1
